# Feature Engineering

## Project Overview
This notebook creates new features from the cleaned Glassdoor Data Science Jobs dataset.

### Objectives
- Load the cleaned dataset
- Extract company age
- Extract job state
- Simplify job titles
- Identify seniority level
- Extract programming languages
- Extract tools & technologies
- Detect remote jobs
- Extract years of experience
- Save the engineered dataset

The engineered dataset generated here will be used for Exploratory Data Analysis and dashboard development.

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)

## Load Clean Dataset

In [3]:
df = pd.read_csv(
    "C:/Users/ASAITHAMBI/OneDrive/Documents/dsjob_eda/data/cleaned_jobs.csv")

df.head()

,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,Per_Hour,Employer_Provided,Min_Salary,Max_Salary,Avg_Salary
0,Data Scientist,53-91,"Data Scientist\nLocation: Albuquerque, NM\nEdu...",3.8,Tecolote Research\n3.8,"Albuquerque, NM","Goleta, CA",501 to 1000 employees,1973,Company - Private,Aerospace & Defense,Aerospace & Defense,$50 to $100 million (USD),-1,0,0,53,91,72.0
1,Healthcare Data Scientist,63-112,What You Will Do:\n\nI. General Summary\n\nThe...,3.4,University of Maryland Medical System\n3.4,"Linthicum, MD","Baltimore, MD",10000+ employees,1984,Other Organization,Health Care Services & Hospitals,Health Care,$2 to $5 billion (USD),-1,0,0,63,112,87.5
2,Data Scientist,80-90,"KnowBe4, Inc. is a high growth information sec...",4.8,KnowBe4\n4.8,"Clearwater, FL","Clearwater, FL",501 to 1000 employees,2010,Company - Private,Security Services,Business Services,$100 to $500 million (USD),-1,0,0,80,90,85.0
3,Data Scientist,56-97,*Organization and Job ID**\nJob ID: 310709\n\n...,3.8,PNNL\n3.8,"Richland, WA","Richland, WA",1001 to 5000 employees,1965,Government,Energy,"Oil, Gas, Energy & Utilities",$500 million to $1 billion (USD),"Oak Ridge National Laboratory, National Renewa...",0,0,56,97,76.5
4,Data Scientist,86-143,Data Scientist\nAffinity Solutions / Marketing...,2.9,Affinity Solutions\n2.9,"New York, NY","New York, NY",51 to 200 employees,1998,Company - Private,Advertising & Marketing,Business Services,Unknown / Non-Applicable,"Commerce Signals, Cardlytics, Yodlee",0,0,86,143,114.5


## Dataset Shape

In [4]:
print(f"Rows : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

Rows : 467
Columns : 19


## Dataset Information

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 467 entries, 0 to 466
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Job Title          467 non-null    str    
 1   Salary Estimate    467 non-null    str    
 2   Job Description    467 non-null    str    
 3   Rating             467 non-null    float64
 4   Company Name       467 non-null    str    
 5   Location           467 non-null    str    
 6   Headquarters       467 non-null    str    
 7   Size               467 non-null    str    
 8   Founded            467 non-null    int64  
 9   Type of ownership  467 non-null    str    
 10  Industry           467 non-null    str    
 11  Sector             467 non-null    str    
 12  Revenue            467 non-null    str    
 13  Competitors        467 non-null    str    
 14  Per_Hour           467 non-null    int64  
 15  Employer_Provided  467 non-null    int64  
 16  Min_Salary         467 non-null    in

## Company Age
Some companies have **Founded = -1**, indicating missing values.

Create a new feature representing company age.

In [6]:
CURRENT_YEAR = 2023

df["Company_Age"] = np.where(
    df["Founded"] > 0,
    CURRENT_YEAR - df["Founded"],
    np.nan
)

Check

In [7]:
df[["Founded", "Company_Age"]].sample(10)

,Founded,Company_Age
320,1965,58.0
169,1984,39.0
315,1875,148.0
458,1997,26.0
388,2008,15.0
202,1981,42.0
77,1997,26.0
326,1965,58.0
397,2004,19.0
387,2005,18.0


## Job State
Extract the state abbreviation from the location column.

Example
```
New York, NY

↓

NY
```

In [8]:
df["Job_State"] = (
    df["Location"]
    .str.split(",")
    .str[-1]
    .str.strip()
)

Check

In [9]:
df[["Location", "Job_State"]].head(10)

,Location,Job_State
0,"Albuquerque, NM",NM
1,"Linthicum, MD",MD
2,"Clearwater, FL",FL
3,"Richland, WA",WA
4,"New York, NY",NY
5,"Dallas, TX",TX
6,"Baltimore, MD",MD
7,"San Jose, CA",CA
8,"Rochester, NY",NY
9,"New York, NY",NY


## Simplify Job Titles
Instead of hundreds of different job titles, group them into common categories.

In [10]:
def simplify_job(title):

    title = title.lower()

    if "data scientist" in title:
        return "Data Scientist"

    elif "data engineer" in title:
        return "Data Engineer"

    elif "data analyst" in title:
        return "Data Analyst"

    elif "machine learning" in title:
        return "ML Engineer"

    elif "manager" in title:
        return "Manager"

    elif "director" in title:
        return "Director"

    else:
        return "Other" 

Apply

In [11]:
df["Job_Simplified"] = (
    df["Job Title"]
    .apply(simplify_job)
)

Check

In [12]:
df["Job_Simplified"].value_counts()

Job_Simplified
Data Scientist    192
Other             101
Data Engineer      75
Data Analyst       68
Manager            12
ML Engineer        11
Director            8
Name: count, dtype: int64

## Seniority Level
Extract seniority from the job title.

In [13]:
def seniority(title):

    title = title.lower()

    if "senior" in title:
        return "Senior"

    elif "lead" in title:
        return "Lead"

    elif "manager" in title:
        return "Manager"

    elif "director" in title:
        return "Director"

    elif "junior" in title:
        return "Junior"

    else:
        return "Not Specified" 

Apply

In [14]:
df["Seniority"] = (
    df["Job Title"]
    .apply(seniority)
)

Check

In [15]:
df["Seniority"].value_counts()

Seniority
Not Specified    358
Senior            73
Lead              13
Manager           13
Director           9
Junior             1
Name: count, dtype: int64

## Programming Languages

Create binary indicator columns to identify whether a job description mentions a specific programming language.

In [27]:
# Programming language patterns
languages = {
    "Python": r"\bPython\b",
    "Java": r"\bJava\b",
    "R": r"\bR\b",
    "SQL": r"\bSQL\b",
    "C++": r"\bC\+\+\b",
    "JavaScript": r"\b(?:JavaScript|JS)\b",
    "Ruby": r"\bRuby\b",
    "Perl": r"\bPerl\b",
    "Scala": r"\bScala\b",
    "Go": r"\bGo\b"
}

Create binary columns.

In [28]:

for language, pattern in languages.items():

    column = language.replace("+", "p") + "_yn"

    df[column] = (
        df["Job Description"]
        .str.contains(
            pattern,
            case=False,
            regex=True,
            na=False
        )
        .astype(int)
    )

Check created columns

In [29]:

df.filter(regex="_yn$").head()

,Python_yn,R_yn,SQL_yn,Java_yn,Cpp_yn,JavaScript_yn,Scala_yn,Go_yn,Ruby_yn,Perl_yn,AWS_yn,Azure_yn,Hadoop_yn,Spark_yn,Docker_yn,Kubernetes_yn,Tableau_yn,Power BI_yn,NLP_yn,Machine Learning_yn,Big Data_yn
0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0
1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,1,1,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1
3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


## Tools & Technologies

Create binary indicator columns for commonly used tools and technologies mentioned in job descriptions.

In [30]:
tools = {
    "AWS": r"\bAWS\b",
    "Azure": r"\bAzure\b",
    "Spark": r"\bSpark\b",
    "Hadoop": r"\bHadoop\b",
    "Docker": r"\bDocker\b",
    "Kubernetes": r"\bKubernetes\b",
    "Tableau": r"\bTableau\b",
    "PowerBI": r"\b(?:Power BI|PowerBI)\b",
    "NLP": r"\bNLP\b",
    "MachineLearning": r"\b(?:Machine Learning|ML)\b",
    "BigData": r"\b(?:Big Data)\b"
}

Create binary columns

In [32]:

for tool, pattern in tools.items():

    df[f"{tool}_yn"] = (
        df["Job Description"]
        .str.contains(
            pattern,
            case=False,
            regex=True,
            na=False
        )
        .astype(int)
    )

## Remote Job Detection

Identify whether a job description mentions remote work.

In [33]:
REMOTE_PATTERN = (
    r"\b(?:remote|work from home|telecommute|virtual)\b"
)

In [34]:
df["Remote_Job"] = (
    df["Job Description"]
    .str.contains(
        REMOTE_PATTERN,
        case=False,
        regex=True,
        na=False
    )
    .astype(int)
)

In [35]:
df["Remote_Job"].value_counts()

Remote_Job
0    447
1     20
Name: count, dtype: int64

## Years of Experience

Extract the first years-of-experience requirement mentioned in each job description

In [36]:
def extract_years_experience(description):

    pattern = r'(\d+)\+?\s*(?:years?|yrs?)'

    match = re.search(
        pattern,
        str(description),
        re.IGNORECASE
    )

    if match:
        return int(match.group(1))

    return np.nan

In [37]:
df["Years_Experience"] = (
    df["Job Description"]
    .apply(extract_years_experience)
)

In [38]:
df["Years_Experience"].head(20)

0     NaN
1     3.0
2     3.0
3     1.0
4     NaN
5     2.0
6     NaN
7     NaN
8     NaN
9     NaN
10    2.0
11    NaN
12    5.0
13    5.0
14    3.0
15    NaN
16    3.0
17    2.0
18    3.0
19    NaN
Name: Years_Experience, dtype: float64

## Experience Groups

Group years of experience into categories for easier analysis.

In [39]:
def experience_group(years):

    if pd.isna(years):
        return "Unknown"

    elif years < 3:
        return "0-2 Years"

    elif years <= 5:
        return "3-5 Years"

    elif years <= 10:
        return "6-10 Years"

    else:
        return "10+ Years"

In [40]:
df["Experience_Group"] = (
    df["Years_Experience"]
    .apply(experience_group)
)

In [41]:
df["Experience_Group"].value_counts()

Experience_Group
3-5 Years     189
Unknown       126
0-2 Years      73
6-10 Years     53
10+ Years      26
Name: count, dtype: int64

## Final Dataset Information

In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 467 entries, 0 to 466
Data columns (total 50 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Job Title            467 non-null    str    
 1   Salary Estimate      467 non-null    str    
 2   Job Description      467 non-null    str    
 3   Rating               467 non-null    float64
 4   Company Name         467 non-null    str    
 5   Location             467 non-null    str    
 6   Headquarters         467 non-null    str    
 7   Size                 467 non-null    str    
 8   Founded              467 non-null    int64  
 9   Type of ownership    467 non-null    str    
 10  Industry             467 non-null    str    
 11  Sector               467 non-null    str    
 12  Revenue              467 non-null    str    
 13  Competitors          467 non-null    str    
 14  Per_Hour             467 non-null    int64  
 15  Employer_Provided    467 non-null    int64  
 16  M

In [43]:
df.isnull().sum()

Job Title                0
Salary Estimate          0
Job Description          0
Rating                   0
Company Name             0
Location                 0
Headquarters             0
Size                     0
Founded                  0
Type of ownership        0
Industry                 0
Sector                   0
Revenue                  0
Competitors              0
Per_Hour                 0
Employer_Provided        0
Min_Salary               0
Max_Salary               0
Avg_Salary               0
Company_Age             33
Job_State                0
Job_Simplified           0
Seniority                0
Python_yn                0
R_yn                     0
SQL_yn                   0
Java_yn                  0
Cpp_yn                   0
JavaScript_yn            0
Scala_yn                 0
Go_yn                    0
Ruby_yn                  0
Perl_yn                  0
AWS_yn                   0
Azure_yn                 0
Hadoop_yn                0
Spark_yn                 0
D

In [44]:
df.head()

,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,Per_Hour,Employer_Provided,Min_Salary,Max_Salary,Avg_Salary,Company_Age,Job_State,Job_Simplified,Seniority,Python_yn,R_yn,SQL_yn,Java_yn,Cpp_yn,JavaScript_yn,Scala_yn,Go_yn,Ruby_yn,Perl_yn,AWS_yn,Azure_yn,Hadoop_yn,Spark_yn,Docker_yn,Kubernetes_yn,Tableau_yn,Power BI_yn,NLP_yn,Machine Learning_yn,Big Data_yn,Years_Experience,PowerBI_yn,MachineLearning_yn,BigData_yn,Remote_Job,Experience_Group
0,Data Scientist,53-91,"Data Scientist\nLocation: Albuquerque, NM\nEdu...",3.8,Tecolote Research\n3.8,"Albuquerque, NM","Goleta, CA",501 to 1000 employees,1973,Company - Private,Aerospace & Defense,Aerospace & Defense,$50 to $100 million (USD),-1,0,0,53,91,72.0,50.0,NM,Data Scientist,Not Specified,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,NaN,1,1,0,0,Unknown
1,Healthcare Data Scientist,63-112,What You Will Do:\n\nI. General Summary\n\nThe...,3.4,University of Maryland Medical System\n3.4,"Linthicum, MD","Baltimore, MD",10000+ employees,1984,Other Organization,Health Care Services & Hospitals,Health Care,$2 to $5 billion (USD),-1,0,0,63,112,87.5,39.0,MD,Data Scientist,Not Specified,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,3.0,0,1,0,0,3-5 Years
2,Data Scientist,80-90,"KnowBe4, Inc. is a high growth information sec...",4.8,KnowBe4\n4.8,"Clearwater, FL","Clearwater, FL",501 to 1000 employees,2010,Company - Private,Security Services,Business Services,$100 to $500 million (USD),-1,0,0,80,90,85.0,13.0,FL,Data Scientist,Not Specified,1,1,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,3.0,0,1,1,0,3-5 Years
3,Data Scientist,56-97,*Organization and Job ID**\nJob ID: 310709\n\n...,3.8,PNNL\n3.8,"Richland, WA","Richland, WA",1001 to 5000 employees,1965,Government,Energy,"Oil, Gas, Energy & Utilities",$500 million to $1 billion (USD),"Oak Ridge National Laboratory, National Renewa...",0,0,56,97,76.5,58.0,WA,Data Scientist,Not Specified,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1.0,0,1,0,0,0-2 Years
4,Data Scientist,86-143,Data Scientist\nAffinity Solutions / Marketing...,2.9,Affinity Solutions\n2.9,"New York, NY","New York, NY",51 to 200 employees,1998,Company - Private,Advertising & Marketing,Business Services,Unknown / Non-Applicable,"Commerce Signals, Cardlytics, Yodlee",0,0,86,143,114.5,25.0,NY,Data Scientist,Not Specified,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,NaN,0,1,0,0,Unknown


Save Feature Engineered Dataset

In [ ]:
df.to_csv(
    "featured_jobs1.csv",
    index=False
)

## Summary
In this notebook, we:
- Calculated company age
- Extracted job state
- Simplified job titles
- Identified seniority levels
- Created binary features for programming languages
- Created binary features for tools and technologies
- Detected remote job postings
- Extracted years of experience required
- Categorized experience into groups
- Saved the engineered dataset for exploratory analysis

The next notebook focuses on Exploratory Data Analysis (EDA) and visualization.